# Serenity Whisper — local demo pipeline

This notebook mirrors a **project-style** ML workflow (dataset manifest → preprocessing → inference),
similar in spirit to how Serenity wires external AI services. **No cloud Whisper API** — weights live under `../models/` after `scripts/download_models.py`.

The **versioned corpus** (what you "see" in the repo) lives under `../dataset/` (`manifest.csv` + `audio/`). That is separate from **model weights** downloaded into `../models/`.


In [ ]:
from pathlib import Path
import csv

ROOT = Path('..').resolve()
manifest = ROOT / 'dataset' / 'manifest.csv'
assert manifest.exists(), (
    'Missing ../dataset/manifest.csv — run from repo: python scripts/build_versioned_dataset.py'
)
with manifest.open(encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
rows[:3]

## Optional: run faster-whisper on the sample clip
Requires `pip install -r ../requirements.txt` and downloaded models.

In [ ]:
from faster_whisper import WhisperModel

model_dir = ROOT / 'models' / 'faster-whisper-tiny.en'
if not model_dir.exists():
    raise SystemExit('Run ../scripts/download_models.py first')

model = WhisperModel(str(model_dir), device='cpu', compute_type='int8')
first = rows[0]
wav = (ROOT / 'dataset' / first['audio_path']).resolve()
assert wav.exists(), wav
segments, info = model.transcribe(str(wav))
print('clip', first['clip_id'], 'wav', wav.name)
print('language', info.language, 'duration', info.duration)
print(''.join(s.text for s in segments))